# 4.10 Eksen İşaretleri Özelleştirme

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/04-matplotlib/10-customizing-ticks.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Customizing Ticks

Matplotlib'ın varsayılan tick konumlandırıcıları (locator) ve biçimlendiricileri (formatter) birçok yaygın durumda genel olarak yeterli olacak şekilde tasarlanmıştır; ancak hiçbir grafik türü için ideal değildir. Bu bölüm, ilgilendiğiniz grafik türü için tick konumlarını ve biçimlendirmesini ayarlamaya yönelik birkaç örnek sunar.

Örneklere geçmeden önce Matplotlib grafiklerinin nesne hiyerarşisinden biraz daha söz edelim. Matplotlib, grafikte görünen her şeyi temsil eden bir Python nesnesine sahip olmayı hedefler: örneğin Figure'ın grafik öğelerinin göründüğü sınır kutusu olduğunu hatırlayın. Her Matplotlib nesnesi alt nesnelerin konteyneri de olabilir: örneğin her Figure bir veya daha fazla Axes içerebilir; her biri de grafik içeriğini temsil eden diğer nesneleri barındırır.

Tick işaretleri de istisna değildir. Her eksenin xaxis ve yaxis öznitelikleri vardır; bunlar da ekseni oluşturan çizgiler, tick'ler ve etiketlerin tüm özelliklerini içeren alt özniteliklere sahiptir.

## Ana ve İkincil Tick'ler

Her eksen içinde ana (major) tick ve ikincil (minor) tick kavramı vardır. Adlarından anlaşılacağı gibi ana tick'ler genelde daha büyük veya belirgindir; ikincil tick'ler daha küçüktür. Varsayılan olarak Matplotlib ikincil tick'leri nadiren kullanır; ancak logaritmik grafiklerde görebilirsiniz (aşağıdaki şekil):


```
import matplotlib.pyplot as plt
plt.style.use('classic')
import numpy as np

%matplotlib inline
```


In [ ]:
# log_axes.py
ax = plt.axes(xscale='log', yscale='log')
ax.set(xlim=(1, 1E3), ylim=(1, 1E3))
ax.grid(True);



Bu grafikte her ana tick büyük bir işaret, etiket ve ızgara çizgisi gösterir; her ikincil tick daha küçük bir işaret gösterir, etiket veya ızgara çizgisi yoktur.

Bu tick özellikleri — konumlar ve etiketler — her eksenin formatter ve locator nesneleri ayarlanarak özelleştirilebilir. Az önce gösterilen grafiğin x ekseni için bunları inceleyelim:


In [ ]:
# print_locators.py
print(ax.xaxis.get_major_locator())
print(ax.xaxis.get_minor_locator())



In [ ]:
# print_formatters.py
print(ax.xaxis.get_major_formatter())
print(ax.xaxis.get_minor_formatter())



Hem ana hem ikincil tick konumlarının bir LogLocator ile belirlendiğini görüyoruz (logaritmik grafik için mantıklı). İkincil tick'lerin etiketleri ise NullFormatter ile biçimlendirilmiş: yani etiket gösterilmeyecek.

Şimdi çeşitli grafikler için bu locator ve formatter'ları ayarlamaya yönelik birkaç örneğe bakalım.

## Tick veya Etiketleri Gizleme

Belki de en yaygın tick/etiket biçimlendirme işlemi tick veya etiketleri gizlemektir. Bu, burada gösterildiği gibi plt.NullLocator ve plt.NullFormatter ile yapılabilir (aşağıdaki şekil):


In [ ]:
# hide_ticks_labels.py
ax = plt.axes()
rng = np.random.default_rng(1701)
ax.plot(rng.random(50))
ax.grid()

ax.yaxis.set_major_locator(plt.NullLocator())
ax.xaxis.set_major_formatter(plt.NullFormatter())



x ekseninden etiketleri (tick/ızgara çizgilerini koruyarak) kaldırdık; y ekseninden tick'leri (dolayısıyla etiket ve ızgara çizgilerini de) kaldırdık. Hiç tick olmaması birçok durumda yararlıdır — örneğin bir görüntü ızgarası göstermek istediğinizde.

Örneğin aşağıdaki şekil, denetimli makine öğrenmesi problemlerinde sık kullanılan farklı yüz görüntülerini içerir (bkz. örneğin Derinlemesine: Destek Vektör Makineleri):


In [ ]:
# face_grid.py
fig, ax = plt.subplots(5, 5, figsize=(5, 5))
fig.subplots_adjust(hspace=0, wspace=0)

# Get some face data from Scikit-Learn
from sklearn.datasets import fetch_olivetti_faces
faces = fetch_olivetti_faces().images

for i in range(5):
    for j in range(5):
        ax[i, j].xaxis.set_major_locator(plt.NullLocator())
        ax[i, j].yaxis.set_major_locator(plt.NullLocator())
        ax[i, j].imshow(faces[10 * i + j], cmap='binary_r')



Her görüntü kendi ekseninde gösterilir; tick konumlandırıcılarını null yaptık çünkü tick değerleri (bu durumda piksel numaraları) bu görselleştirme için anlamlı bilgi taşımaz.

## Tick Sayısını Azaltma veya Artırma

Varsayılan ayarlarla yaygın bir sorun, küçük alt grafiklerde etiketlerin kalabalıklaşmasıdır. Aşağıdaki grafik ızgarasında bunu görebiliriz (aşağıdaki şekil):


In [ ]:
# subplots_4x4.py
fig, ax = plt.subplots(4, 4, sharex=True, sharey=True)



Özellikle x ekseni tick'lerinde sayılar neredeyse üst üste binerek okunması zor hale gelir. Bunu plt.MaxNLocator ile ayarlayabiliriz; gösterilecek maksimum tick sayısını belirtmemize izin verir. Bu sayıya göre Matplotlib uygun tick konumlarını seçer (aşağıdaki şekil):


In [ ]:
# For every axis, set the x and y major locator
for axi in ax.flat:
    axi.xaxis.set_major_locator(plt.MaxNLocator(3))
    axi.yaxis.set_major_locator(plt.MaxNLocator(3))
fig



Bu işleri çok daha temiz hale getirir. Düzenli aralıklı tick konumları üzerinde daha fazla kontrol istiyorsanız bir sonraki bölümde ele alacağımız plt.MultipleLocator'ı da kullanabilirsiniz.

## Gelişmiş Tick Biçimlendirme

Matplotlib'ın varsayılan tick biçimlendirmesi çoğu zaman yetersiz kalabilir: geniş bir varsayılan olarak iyi çalışır, ancak bazen farklı bir şey istersiniz. Aşağıdaki sinüs ve kosinüs eğrisi grafiğini düşünün (aşağıdaki şekil):


In [ ]:
# Plot a sine and cosine curve
fig, ax = plt.subplots()
x = np.linspace(0, 3 * np.pi, 1000)
ax.plot(x, np.sin(x), lw=3, label='Sine')
ax.plot(x, np.cos(x), lw=3, label='Cosine')

# Set up grid, legend, and limits
ax.grid(True)
ax.legend(frameon=False)
ax.axis('equal')
ax.set_xlim(0, 3 * np.pi);



Burada birkaç değişiklik yapmak isteyebiliriz. İlk olarak, bu veri için tick ve ızgara çizgilerini $\pi$ katlarında aralamak daha doğaldır. Bunu, verdiğimiz sayının katlarında tick konumlandıran MultipleLocator ile yapabiliriz. İyi ölçüde hem $\pi/2$ hem $\pi/4$ katlarında ana ve ikincil tick ekleyelim (aşağıdaki şekil):


In [ ]:
# multiple_locator_pi.py
ax.xaxis.set_major_locator(plt.MultipleLocator(np.pi / 2))
ax.xaxis.set_minor_locator(plt.MultipleLocator(np.pi / 4))
fig



Ancak şimdi tick etiketleri biraz saçma görünüyor: $\pi$ katları olduklarını görebiliyoruz, ancak ondalık gösterim bunu hemen iletmiyor. Bunu tick biçimlendiricisini değiştirerek düzeltebiliriz. İstediğimiz için yerleşik bir biçimlendirici yok; bunun yerine tick çıktıları üzerinde ince kontrol sağlayan kullanıcı tanımlı bir fonksiyon kabul eden plt.FuncFormatter kullanacağız (aşağıdaki şekil):


In [ ]:
# func_formatter_pi.py
def format_func(value, tick_number):
    # find number of multiples of pi/2
    N = int(np.round(2 * value / np.pi))
    if N == 0:
        return "0"
    elif N == 1:
        return r"$\pi/2$"
    elif N == 2:
        return r"$\pi$"
    elif N % 2 > 0:
        return rf"${N}\pi/2$"
    else:
        return rf"${N // 2}\pi$"

ax.xaxis.set_major_formatter(plt.FuncFormatter(format_func))
fig



Bu çok daha iyi! Dizeyi dolar işaretleri içine alarak Matplotlib'ın LaTeX desteğinden yararlandık. Matematiksel sembol ve formüllerin gösterimi için çok uygundur: bu durumda "$\pi$" Yunan harfi $\pi$ olarak işlenir.

## Biçimlendiriciler ve Konumlandırıcılar Özeti

Mevcut birkaç biçimlendirici ve konumlandırıcıyı gördük; bu bölümü, yerleşik locator ve formatter seçeneklerinin kısa listesiyle bitireceğim. Ayrıntılar için docstring'lere veya Matplotlib çevrimiçi dokümantasyonuna bakın. Aşağıdakilerin her biri plt ad alanında kullanılabilir:

Bu seçeneklerin daha fazla örneğini kitabın geri kalanında göreceğiz.

> **Not**
>

### 🧪 Şimdi deneyin

🧪 π ekseninde tick
      Basit bir sinüs grafiğinde x eksenini π katlarında işaretleyin:
          
      import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots()
ax.plot(x, np.sin(x))
ax.xaxis.set_major_locator(plt.MultipleLocator(np.pi / 2))
ax.set_xlim(0, 2 * np.pi)
fig

> **Not**
>
